# **Deeploc 2.1**

2025 겨울 URP / 문서연, 김대현, 권효재


---


여기서부턴 Deeploc 2.1과 같은 구조로 학습을 진행!


---


개선/공부 필요(2026_01_18):

---

# **라이브러리 설명**

# torch
Pytorch 라이프러리 패키지


* torch.autograd: 자동 미분을 위한 함수가 포함됨(ex. enable/no_grad: 자동 미분 on/off, Function: 자체 미분 함수 정의 클래스)
* torch.nn: 신경망 구축을 위한 기본 데이터 구조/레이어(RNN/LSTM)/활성화 함수(ReLU)/손실 함수(MSELoss) 포함됨
* torch.optim: 확률적 경사 하강법(Stochastic Gradient Descent, SGD) 중심의 파라미터 옵티마이저 알고리즘
* torch.utils.data: SDG 반복연산 시에 사용하는 미니배치 유틸리티 포함됨
* torch.onnx: ONNX(Open Neural Network Exchange) 포맷으로 모델 export 할 때 사용

In [11]:
# 필요 라이브러리 설치 및 import
#!pip install -q torch pandas numpy safetensors

# from torchvision.ops import sigmoid_focal_loss as FocalLoss

import gc
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import math

from safetensors import safe_open
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [12]:
# <저장소 설정>

# Colab 환경
# from google.colab import drive

# drive.mount("/content/drive")
# SAVE_PATH = (
#    "/content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding_safetensors"
# )

# 로컬 환경
SAVE_PATH = '../ESMC_embedding_safetensors'

os.makedirs(SAVE_PATH, exist_ok=True)

In [13]:
# 데이터셋 클래스 정의
class Dataset(Dataset):
    # 데이터셋 전처리
    def __init__(self, save_path, save_path_x, save_path_y):
        self.save_path = save_path
        self.key_list = []
        self.targets = {}
        self.path_x = os.path.join(self.save_path, save_path_x)
        self.path_y = save_path_y

        for k in self.path_y:
            data = torch.load(os.path.join(self.save_path, k), weights_only=False)
            self.targets.update(data)
            self.key_list.extend(data.keys())

    # 몇개있는지 알려줌
    def __len__(self):
        return len(self.key_list)

    # 데이터셋 샘플 1개 가져오기
    def __getitem__(self, idx):
        key = self.key_list[idx]
        with safe_open(self.path_x, framework="pt", device='cpu') as f:
            embedding = f.get_tensor(key).clone()
            embedding = embedding.squeeze(0)
        target_numpy = self.targets[key]
        target = torch.from_numpy(target_numpy).float()
        return embedding, target

In [14]:
def padding_collate_fn(batch):
    #임베딩/타겟 분리
    embeddings = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    # 2. 임베딩 패딩
    padded_embeddings = pad_sequence(embeddings, batch_first=True, padding_value=0)
    return padded_embeddings, targets

In [ ]:
class model(nn.Module):
    def __init__(self, embedding_dim=1152, attn_dim=128, output_dim = 4, heads_nums=2):
        super(model, self).__init__()
        self.input_layer_normalization = nn.LayerNorm(embedding_dim)
        self.input_linear_fc = nn.Linear(embedding_dim, attn_dim)
        self.attention_layer_normalization = nn.LayerNorm(attn_dim)
        self.attention_head = MultiheadAttentionPooling(attn_dim=attn_dim, heads_nums=heads_nums)
        self.dropout = nn.Dropout(0.1)
        self.output_linear_fc = nn.Linear(attn_dim, output_dim)  # mlp층 쌓으면 attn_dim 아니고 hiddden_dim 됨 이거 나중에 고쳐야할듯
    def forward(self, x):
        #layer normalization, 선형층 통과 > embedding dim에서 attn dim으로,,
        x = self.input_layer_normalization(x)
        x = self.input_linear_fc(x)
        #attention head 통과 > (batch_size, attn_dim)
        x = self.attention_layer_normalization(x)
        x = self.attention_head(x)
        x = self.dropout(x)
        #선형층 통과(분류기) > attn dim에서 4개로,,
        x = self.output_linear_fc(x)
        return x

In [15]:
# 앞에서 Layer normalization 해준걸 인풋으로 받는다고 가정
class MultiheadAttentionPooling(nn.Module):
    def __init__(self, attn_dim=128, heads_nums=2, kernel_size=5):
        super().__init__()

        self.attn_dim = attn_dim
        # 우선은,, 멀티헤드로 구현을 한다
        self.heads_nums = heads_nums
        # 멀티헤드의 디멘션은 전체 디멘션을 헤드 개수로 나눈것
        # 이거때문에 헤드 개수를 잘 나눠지도록(?) 설정함 보통
        self.head_dim = attn_dim // heads_nums

        # Q, K = V 만들기
        # Q : learnable Query, 먼저 (1, 1, attn_dim) 에 해당하는 빈 벡터 > xavier_uniform_ 하면 입출력 고려해서 난수생성 가능
        self.query = nn.Parameter(torch.empty(1, 1, attn_dim))
        nn.init.xavier_uniform_(self.query.data)
        self.w_kv = nn.Linear(
            attn_dim, attn_dim
        )  # 입력/출력 크기가 attn_dim인 Linear FC를 수행하는 모듈 // key value 짜피 같으니까 이걸로 한번에 할거고 논문도 그렇게 했는데 둘이 따로 초기화한다면? 즉 파라미터가 두개라면..?

    def forward(self, x):
        # 입력으로 받을 형태: (batch_size, sequence_length, attn_dim)
        batch_size, sequence_length, _ = x.shape

        # 배치 사이즈에 맞게 복제: 각 배치 샘플 전부 같은 쿼리 파라미터 공유
        # attn_dim을 헤드별로 쪼갬()
        # 헤드별로 계산할거라서 헤드를 앞으로 뺌
        Q = (
            self.query.repeat(batch_size, 1, 1)
            .view(batch_size, 1, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        K = (
            self.w_kv(x)
            .view(batch_size, sequence_length, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        V = (
            self.w_kv(x)
            .view(batch_size, sequence_length, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        # 셋다 (batch_size, head_nums, sequence_length(Q:1; per token), head_dim)
        # print(Q)
        # print(K)
        # print(V)
        # 행렬곱 > Attention Scalar score 구함!
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        print(f"어텐션 Score: {scores.shape}")
        # Conv1d 가우시안 필터
        scores_c1d_gaussian = scores

        # Softmax 적용
        attn_weights = F.softmax(scores_c1d_gaussian, dim=-1)
        print(f"어텐션 Weights: {attn_weights.shape}")

        # 행렬곱 > weighted attention
        attnpooled_nosum = torch.matmul(attn_weights, V)
        print(f"어텐션 Pooling (합하기 전): {attnpooled_nosum.shape}")

        # 가중합을 위해 (batch_size, head_nums, 1, head_dim) 순서니까
        # 1 없애고 reshape로 head concat해주기
        attentionpooled = attnpooled_nosum.squeeze(2).reshape(batch_size, self.attn_dim)
        print(f"어텐션 Pooling: {attentionpooled.shape}")

        return attentionpooled

In [16]:
def sigmoid_focal_loss(
    inputs: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 0.25,
    gamma: float = 2,
    reduction: str = "none",
) -> torch.Tensor:

    if not (0 <= alpha <= 1) and alpha != -1:
        raise ValueError(
            f"Invalid alpha value: {alpha}. alpha must be in the range [0,1] or -1 for ignore."
        )

    p = torch.sigmoid(inputs)
    ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    loss = ce_loss * ((1 - p_t) ** gamma)

    if alpha >= 0:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss

    # Check reduction option and return loss accordingly
    if reduction == "none":
        pass
    elif reduction == "mean":
        loss = loss.mean()
    elif reduction == "sum":
        loss = loss.sum()
    else:
        raise ValueError(
            f"Invalid Value for arg 'reduction': '{reduction} \n Supported reduction modes: 'none', 'mean', 'sum'"
        )
    return loss

In [ ]:
# k fold CV를 위한 반복문 코드
# k를 숫자 변수로 하나 받고
# range(k) 리스트에서 한번씩 빼는 식으로 하면 되겟다!!!
# 나중에 데이터 인풋을 이런 느낌으로 넣어서 4 폴드 교차검증 돌릴거임
list = range(K_CV)
for i in list:
    SAVE_PATH_TARGETS = [f'targets_part_{j}.pt' for j in list if j != i]
    SAVE_PATH_EMBEDDINGS = [f'embeddings_part_{j}.safetensors' for j in list if j != i]
    print(SAVE_PATH_TARGETS, SAVE_PATH_EMBEDDINGS)
    for k in SAVE_PATH_TARGETS:
        a = torch.load(os.path.join(SAVE_PATH, k), weights_only=False)
    print(a.keys())

['targets_part_1.pt', 'targets_part_2.pt', 'targets_part_3.pt'] ['embeddings_part_1.safetensors', 'embeddings_part_2.safetensors', 'embeddings_part_3.safetensors']
['targets_part_0.pt', 'targets_part_2.pt', 'targets_part_3.pt'] ['embeddings_part_0.safetensors', 'embeddings_part_2.safetensors', 'embeddings_part_3.safetensors']
['targets_part_0.pt', 'targets_part_1.pt', 'targets_part_3.pt'] ['embeddings_part_0.safetensors', 'embeddings_part_1.safetensors', 'embeddings_part_3.safetensors']
['targets_part_0.pt', 'targets_part_1.pt', 'targets_part_2.pt'] ['embeddings_part_0.safetensors', 'embeddings_part_1.safetensors', 'embeddings_part_2.safetensors']


In [ ]:
test_tensor = "embeddings_part_3.safetensors"
test_target = ["targets_part_3.pt"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET = dataset(SAVE_PATH, test_tensor, test_target)
LOADER = DataLoader(DATASET, batch_size=5, collate_fn=padding_collate_fn)
MODEL = model(embedding_dim=1152, attn_dim=128, output_dim=4, heads_nums=2).to(DEVICE)
NUM_EPOCHS = 10
OPTIMIZER = AdamW(MODEL.parameters(), lr=1e-4)
for epoch in range(NUM_EPOCHS):
    model = MODEL.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for embeddings, targets in LOADER:
        OPTIMIZER.zero_grad()

        outputs = model(embeddings)
        targets = torch.stack(targets).to(outputs.device)

        loss_value = sigmoid_focal_loss(outputs, targets, alpha=0.25, gamma=2, reduction="mean")
        loss_value.backward()
        OPTIMIZER.step()

        running_loss += loss_value.item() * embeddings.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        correct += (predicted == targets).sum().item()
        total += targets.numel()
    epoch_loss = running_loss / len(DATASET)
    epoch_acc = correct / total

    model.eval()  # 모델을 평가 모드로 전환
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    # with torch.no_grad(): # 평가 땐 기울기 계산 비활성화 (메모리 절약)
    #     for inputs, labels in val_dataloader: # 검증용 로더 (없으면 생략 가능)
    #         inputs = inputs.float().to(DEVICE)
    #         labels = labels.long().to(DEVICE)

    #         outputs = model(inputs)
    #         loss = criterion(outputs, labels)

    #         val_loss += loss.item()
    #         _, predicted = torch.max(outputs, 1)
    #         val_total += labels.size(0)
    #         val_correct += (predicted == labels).sum().item()

    # val_loss = val_loss / len(val_dataloader)
    # val_acc = 100 * val_correct / val_total

    # print(f"Epoch [{epoch+1}/{num_epochs}] "
    #       f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
    #       f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")